# NER gia phả — huấn luyện 2 giai đoạn (học trước trên nhãn Gemini → tinh chỉnh bằng nhãn người), kiểm chéo 5 lượt
**Tốn tiền (GPU Colab, ~1–1,5 giờ).** Cần trong `MyDrive/family_tree_ner/`: `train_ner.py`, `vi_cv5.zip`, `vi_silver.zip`. Cùng 5 lượt chia với lần kiểm chéo trước nên so sánh trực tiếp được (lần trước gộp F1 = 0,538).

In [ ]:
MODEL = "vinai/phobert-base"
BASE = "/content/drive/MyDrive/family_tree_ner"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!pip -q install "transformers>=4.40"
import torch; assert torch.cuda.is_available(), "Chưa bật GPU"
!cd /content && unzip -qo {BASE}/vi_cv5.zip && unzip -qo {BASE}/vi_silver.zip && ls

In [ ]:
# Giai đoạn 1: học trước trên nhãn Gemini (không chứa 25 cây người soát)
!python {BASE}/train_ner.py --model {MODEL} --data-dir /content/vi_silver --output-dir /content/out_silver --epochs 6 --lr 5e-5 --batch-size 16 2>&1 | grep -E "epoch|thiết bị"
!ls /content/out_silver

In [ ]:
# Giai đoạn 2: tinh chỉnh bằng nhãn người, 5 lượt, bắt đầu từ model giai đoạn 1
import subprocess, json, shutil, os
os.makedirs(f"{BASE}/cv_results_2stage", exist_ok=True)
for k in range(5):
    out = f"/content/out2/fold{k}"
    subprocess.run(["python", f"{BASE}/train_ner.py", "--model", "/content/out_silver", "--data-dir", f"/content/vi_cv5/fold{k}",
                    "--output-dir", out, "--test-split", "test", "--epochs", "15", "--lr", "3e-5", "--batch-size", "16"], check=True)
    shutil.copy(f"{out}/model_card.json", f"{BASE}/cv_results_2stage/fold{k}_model_card.json")
    shutil.rmtree(out)
    c = json.load(open(f"{BASE}/cv_results_2stage/fold{k}_model_card.json"))
    print(f"== fold{k}: best_epoch={c['best_epoch']} dev_F1={c['dev']['micro']['f1']} test_F1={c['test']['micro']['f1']}")

In [ ]:
from collections import Counter
tot = {"micro": Counter(), "labels": {}}
for k in range(5):
    c = json.load(open(f"{BASE}/cv_results_2stage/fold{k}_model_card.json"))["test"]
    for key in ("tp","fp","fn"): tot["micro"][key] += c["micro"][key]
    for l, v in c["per_label"].items():
        d = tot["labels"].setdefault(l, Counter())
        for key in ("tp","fp","fn"): d[key] += v[key]
def prf(d):
    p = d["tp"]/(d["tp"]+d["fp"]) if d["tp"]+d["fp"] else 0; r = d["tp"]/(d["tp"]+d["fn"]) if d["tp"]+d["fn"] else 0
    return dict(P=round(p,3), R=round(r,3), F1=round(2*p*r/(p+r),3) if p+r else 0, **d)
print("GỘP 25 CÂY (2 giai đoạn):", prf(tot["micro"]))
for l, d in tot["labels"].items(): print(l, prf(d))
json.dump({"pooled": prf(tot["micro"]), "per_label": {l: prf(d) for l, d in tot["labels"].items()}}, open(f"{BASE}/cv_results_2stage/pooled.json","w"), indent=1)

In [ ]:
from google.colab import runtime
runtime.unassign()   # NGẮT RUNTIME để dừng tính tiền